In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

In [0]:
# =========================
# Dirty Customers Dataset
# =========================

customers_data = [
    (1, "John Doe", "john@example.com", "Hyderabad"),
    (2, "Alice ", "alice@example.com", "Chennai"),
    (3, None, "bob@example.com", "Bangalore"),        # NULL name
    (4, "David", None, "Mumbai"),                    # NULL email
    (5, "Eva", "eva@example.com", "Hyderabad"),
    (6, "Frank", "frank@example.com", "Delhi"),
]

customers = spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city"])


In [0]:
# =========================
# Dirty Orders Dataset
# =========================

orders_data = [
    (101, 1, "2024-01-01", 1000),
    (102, 2, "2024-01-02", 2000),
    (103, 3, "2024-01-03", -500),     # INVALID negative value
    (104, 99, "2024-01-04", 1500),    # INVALID FK (customer_id 99)
    (105, 1, "2024-01-05", None),     # NULL amount
    (106, 5, "2024-01-06", 3000),
    (107, 5, "2024-01-07", 3000),     # duplicate-like record
]

orders = spark.createDataFrame(orders_data, ["order_id", "customer_id", "order_date", "amount"])

In [0]:
# =========================
# Convert date column
# =========================

orders = orders.withColumn("order_date", to_date(col("order_date")))

In [0]:
customers.show()
orders.show()

+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|  Alice |alice@example.com|  Chennai|
|          3|    NULL|  bob@example.com|Bangalore|
|          4|   David|             NULL|   Mumbai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2024-01-01|  1000|
|     102|          2|2024-01-02|  2000|
|     103|          3|2024-01-03|  -500|
|     104|         99|2024-01-04|  1500|
|     105|          1|2024-01-05|  NULL|
|     106|          5|2024-01-06|  3000|
|     107|          5|2024-01-07|  3000|
+--------+-----------+----------+------+



In [0]:
# TODO 1: Clean data
# - Remove nulls
# - Handle negative values
# - Trim names
# Remove nulls in important columns
customers_clean = customers.dropna(subset=["name", "email"])
orders_clean = orders.dropna(subset=["amount"])
customers_clean.show()
orders_clean.show()

+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|  Alice |alice@example.com|  Chennai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2024-01-01|  1000|
|     102|          2|2024-01-02|  2000|
|     103|          3|2024-01-03|  -500|
|     104|         99|2024-01-04|  1500|
|     106|          5|2024-01-06|  3000|
|     107|          5|2024-01-07|  3000|
+--------+-----------+----------+------+



In [0]:
orders_clean = orders_clean.withColumn(
    "amount",
    when(col("amount") < 0, 0).otherwise(col("amount"))
)
orders_clean.show()

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2024-01-01|  1000|
|     102|          2|2024-01-02|  2000|
|     103|          3|2024-01-03|     0|
|     104|         99|2024-01-04|  1500|
|     106|          5|2024-01-06|  3000|
|     107|          5|2024-01-07|  3000|
+--------+-----------+----------+------+



In [0]:
customers_clean=customers_clean.withColumn("name", trim(col("name")))
customers_clean.show()

+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|   Alice|alice@example.com|  Chennai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+



In [0]:
# TODO 2: Validate data
# - Find invalid customer_id using left_anti join
invalid_customers=orders_clean.join(customers_clean,"customer_id","left_anti")
invalid_customers.show()

+-----------+--------+----------+------+
|customer_id|order_id|order_date|amount|
+-----------+--------+----------+------+
|          3|     103|2024-01-03|  -500|
|         99|     104|2024-01-04|  1500|
+-----------+--------+----------+------+



In [0]:
Joined_data=customers_clean.join(orders_clean,"customer_id")
Joined_data.show()

+-----------+--------+-----------------+---------+--------+----------+------+
|customer_id|    name|            email|     city|order_id|order_date|amount|
+-----------+--------+-----------------+---------+--------+----------+------+
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|
+-----------+--------+-----------------+---------+--------+----------+------+



In [0]:
Total_spent=Joined_data.select("*").groupBy("customer_id").agg(sum("amount").alias("Total_spent"))
Total_spent.show()

+-----------+-----------+
|customer_id|Total_spent|
+-----------+-----------+
|          1|       1000|
|          2|       2000|
|          5|       6000|
+-----------+-----------+



In [0]:
Joined_data.show()

+-----------+--------+-----------------+---------+--------+----------+------+
|customer_id|    name|            email|     city|order_id|order_date|amount|
+-----------+--------+-----------------+---------+--------+----------+------+
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|
+-----------+--------+-----------------+---------+--------+----------+------+



In [0]:
Joined_data=Joined_data.join(Total_spent,"customer_id")
Joined_data.show()

+-----------+--------+-----------------+---------+--------+----------+------+-----------+
|customer_id|    name|            email|     city|order_id|order_date|amount|Total_spent|
+-----------+--------+-----------------+---------+--------+----------+------+-----------+
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|       1000|
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|       2000|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|       6000|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|       6000|
+-----------+--------+-----------------+---------+--------+----------+------+-----------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank



In [0]:
window_spec = Window.orderBy(col("total_spent").desc())

ranked_customers = Joined_data.withColumn(
    "rank",
    rank().over(window_spec)
)

ranked_customers.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------+-----------------+---------+--------+----------+------+-----------+----+
|customer_id|    name|            email|     city|order_id|order_date|amount|Total_spent|rank|
+-----------+--------+-----------------+---------+--------+----------+------+-----------+----+
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|       6000|   1|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|       6000|   1|
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|       2000|   3|
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|       1000|   4|
+-----------+--------+-----------------+---------+--------+----------+------+-----------+----+



In [0]:
customers_clean.show()
orders_clean.show()
Joined_data.show()

+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|   Alice|alice@example.com|  Chennai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2024-01-01|  1000|
|     102|          2|2024-01-02|  2000|
|     103|          3|2024-01-03|  -500|
|     104|         99|2024-01-04|  1500|
|     106|          5|2024-01-06|  3000|
|     107|          5|2024-01-07|  3000|
+--------+-----------+----------+------+

+-----------+--------+-----------------+---------+--------+----------+------+-----------+
|customer_id|    name|            email|     city|order_id|order_date|amount|Total_spent|


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

window_spec = Window.partitionBy("city") \
    .orderBy(col("Total_spent").desc())

ranked_df = Joined_data.withColumn(
    "rank",
    dense_rank().over(window_spec)
)
ranked_df.show()

+-----------+--------+-----------------+---------+--------+----------+------+-----------+----+
|customer_id|    name|            email|     city|order_id|order_date|amount|Total_spent|rank|
+-----------+--------+-----------------+---------+--------+----------+------+-----------+----+
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|       2000|   1|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|       6000|   1|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|       6000|   1|
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|       1000|   2|
+-----------+--------+-----------------+---------+--------+----------+------+-----------+----+



In [0]:
Joined_data = Joined_data.withColumn(
    "month",
    month(col("order_date"))
)
Joined_data.show()

+-----------+--------+-----------------+---------+--------+----------+------+-----------+-----+
|customer_id|    name|            email|     city|order_id|order_date|amount|Total_spent|month|
+-----------+--------+-----------------+---------+--------+----------+------+-----------+-----+
|          1|John Doe| john@example.com|Hyderabad|     101|2024-01-01|  1000|       1000|    1|
|          2|   Alice|alice@example.com|  Chennai|     102|2024-01-02|  2000|       2000|    1|
|          5|     Eva|  eva@example.com|Hyderabad|     107|2024-01-07|  3000|       6000|    1|
|          5|     Eva|  eva@example.com|Hyderabad|     106|2024-01-06|  3000|       6000|    1|
+-----------+--------+-----------------+---------+--------+----------+------+-----------+-----+



In [0]:
Joined_data.groupBy("month").agg(sum(col("amount"))).show()

+-----+-----------+
|month|sum(amount)|
+-----+-----------+
|    1|       9000|
+-----+-----------+

